# 2025-10-20: Add in corrected labels to build out the raw object
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology
**Main aim**: Attach the finalized corrected cell type labels (from notebook 17) onto the raw BMMC AnnData object. Adds CMV serostatus, computes patient age from draw date and birth year, maps visit timepoints to abbreviated labels, and drops cells that failed QC (no assigned L1 label). Saves the publication-ready raw object and metadata.

In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import scanpy.external as sce

sc.settings.n_jobs = 50
sc.settings.verbosity = 0

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
adata = sc.read_h5ad('../../../data/rna/raw-files/all-bmmc-raw.h5ad')

In [3]:
# CMV serostatus per subject (source: clinical metadata)
cmv_df = pd.read_csv('../../../data/rna/metadata/subject_cmv_status.csv')
cmv_dict = dict(zip(cmv_df['subject_id'], cmv_df['cmv_status']))
adata.obs["subject.cmv"] = adata.obs["subject.subjectGuid"].map(cmv_dict)

In [4]:
all_labels = pd.read_parquet('../../../data/rna/bmmc-labels/final-bmmc-labels.parquet')
adata.obs = adata.obs.merge(all_labels, on='barcodes', how='left')

#### Here, we're cleaning up the data by removing all cell types that dont have assigned labels
> These cells failed QC upstream

In [5]:
visit_labels = {
    "MM Pre-Treatment": "PreTx",
    "MM Post Induction 2-Cycles": "PI2C",
    "MM End Induction 1st Draw": "EI",
    "MM Post Transplant 60 Days": "ASCT60d",
    "MM Post Transplant 90 Days": "ASCT90d",
    "MM Post Transplant 1 year": "ASCT1y",
    "MM Post Transplant 2 year": "ASCT2y",
    "Healthy": "Healthy"
}

adata.obs['label.visitDetails'] = adata.obs['sample.visitDetails'].map(
    visit_labels).astype('category').cat.remove_unused_categories()

In [6]:
adata = adata[adata.obs['aifi_celltype_l1'].notna()]
adata.raw = adata

In [7]:
adata.obs['sample.drawDate'] = pd.to_datetime(adata.obs['sample.drawDate'])
adata.obs['subject.birthYear'] = adata.obs['subject.birthYear'].astype(int)

# compute age
adata.obs['subject.age'] = (
    adata.obs['sample.drawDate'].dt.year - adata.obs['subject.birthYear']
)
adata.obs['sample.drawDate'] = adata.obs['sample.drawDate'].astype(str)

In [8]:
adata.write('../../../data/rna/final-objects/final-bmmc-raw.h5ad')
adata.obs.to_parquet('../../../data/rna/final-objects/final-bmmc-metadata.parquet')